# LightGCN cross-media training (Kaggle GPU)

Trains + evaluates LightGCN on an Amazon Reviews 2023 category using a free Kaggle **T4 GPU**, then hands you a small `.npz` embeddings file to download and drop into the project's `models/graph/artifacts/`.

**Before you start:**
1. **Settings -> Accelerator = `GPU T4 x2`** (NOT P100 - Kaggle's PyTorch build dropped support for the P100's sm_60 architecture). **Settings -> Internet = On**. Both need a phone-verified Kaggle account.
2. Make sure your GitHub repo has the latest code. From your laptop:
   ```
   git add models/ scripts/ notebooks/
   git commit -m "Update LightGCN benchmark pipeline"
   git push
   ```
   (Cell 2 pulls the latest `origin/main` on every run, so you never need to re-clone by hand.)

**How to run it - this matters.** Use **Save Version -> Save & Run All (Commit)**, *not* the interactive Run All.
A committed run executes headless on Kaggle's servers with up to a 12-hour budget and keeps going after you
close the tab. An interactive session is tied to your browser and Kaggle reclaims it once the tab is idle or
disconnected, which is the usual reason a long run "stops itself" partway through. Check progress under the
notebook's **Versions** tab; outputs land in that version's **Output** tab.

Rough budget at `MAX_USERS = 40000`: preprocessing once (a few minutes), then two 60-epoch trainings
(holdout eval + final artifact). Every stage prints an elapsed/ETA line every 5 epochs, so if the log is
silent for more than a couple of minutes something is genuinely wrong.

To train the other domains later, change `CATEGORY` in the config cell and run again.

**Reading the results:** Amazon data is very sparse (~0.04% density), so absolute numbers are small. Recall@20 around **0.03-0.08** is normal and publishable. What matters is that LightGCN (Step 4) clearly beats the popularity baseline (Step 3).

In [ ]:
# ---------------- CONFIG (edit these) ----------------
REPO_URL   = "https://github.com/sanjay-pokee/Real-Time-Adaptive-Cross-Media-Recommendation-System.git"

CATEGORY   = "Movies_and_TV"   # then rerun with "Books", "CDs_and_Vinyl", "Industrial_and_Scientific", ...
MAX_USERS  = 40000             # 40000 = fast sanity run. 200000 = credible paper number.
EPOCHS     = 60
BATCH_SIZE = 4096             # bigger batch = far fewer full-graph propagations = much faster
USER_CORE  = 10
ITEM_CORE  = 10
EVAL_K     = 20
EVAL_BATCH = 2048             # fine on the T4's 16 GB; lower to 256 only if you hit out-of-memory
# ----------------------------------------------------

ART_NAME  = f"lightgcn_amazon_{CATEGORY.lower()}.npz"
# Steps 2-5 all read this one prepared file, so the CSV parse + k-core filter
# happen once per run instead of once per step.
PREPARED  = f"/kaggle/working/prepared_{CATEGORY.lower()}_{MAX_USERS}.parquet"
print("category:", CATEGORY, "| max_users:", MAX_USERS, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE)

In [ ]:
# Clone (or update) the repo, install the deps Kaggle doesn't ship, check the GPU.
import os, subprocess, sys

WORK = "/kaggle/working"
REPO_DIR = os.path.join(WORK, "repo")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Re-run: pull whatever you last pushed. datasets/ is gitignored, so the
    # already-downloaded CSV survives and does not need re-downloading.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    print("repo updated to latest origin/main")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("repo cloned")
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pandas", "pyarrow"], check=True)


def run_step(*args):
    """Run a pipeline step as a child process with UNBUFFERED output.

    Without `-u` the child's stdout is block-buffered (a notebook is a pipe, not a
    tty), so nothing appears until the process exits and a 40-minute step is
    indistinguishable from a hang. Keep the flag.
    """
    subprocess.run([sys.executable, "-u", "-m", *args], check=True)


import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 7:
    print("WARNING: this GPU is too old for Kaggle's PyTorch build. "
          "Switch Settings -> Accelerator to 'GPU T4 x2'.")

# Sanity check: the repo must contain the current benchmark-dataset loader.
assert os.path.isfile("models/graph/datasets.py"), (
    "models/graph/datasets.py missing -> your pushed GitHub code is stale. "
    "Commit + push from your laptop, then re-run this cell."
)
from models.graph.datasets import is_prepared_cache  # noqa: F401  (fails loudly if the repo is stale)
print("repo code OK")

## Step 1 - download the category file from HuggingFace

In [ ]:
run_step("scripts.fetch_amazon_dataset", "--category", CATEGORY)

import glob
matches = glob.glob(f"datasets/amazon/**/{CATEGORY}.csv", recursive=True)
assert matches, "download did not produce a CSV"
DATA_PATH = matches[0]
print("data file:", DATA_PATH)

## Step 2 - filter the graph once and cache it

Reads the CSV, keeps ratings >= 4 as positives, applies the k-core filter and the user subsample, then writes
the result to `PREPARED`. Steps 3-5 load that parquet directly, so this cost is paid once per run instead of
once per step.

In [ ]:
run_step("models.graph.datasets", DATA_PATH,
         "--user-core", str(USER_CORE), "--item-core", str(ITEM_CORE),
         "--max-users", str(MAX_USERS),
         "--out", PREPARED)

## Step 3 - most-popular baseline (the number LightGCN must beat)

In [ ]:
run_step("models.graph.evaluate_lightgcn",
         "--dataset", PREPARED,
         "--baseline", "popularity", "--k", str(EVAL_K))

## Step 4 - train + evaluate LightGCN (holdout metrics)

In [ ]:
run_step("models.graph.evaluate_lightgcn",
         "--dataset", PREPARED,
         "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
         "--k", str(EVAL_K), "--eval-batch-size", str(EVAL_BATCH))

## Step 5 - train the final artifact on ALL kept interactions and save it

In [ ]:
import os
OUT = f"/kaggle/working/{ART_NAME}"
run_step("models.graph.train_lightgcn",
         "--dataset", PREPARED,
         "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
         "--output", OUT)

for ext in (".npz", ".json"):
    artifact = OUT[:-4] + ext
    if os.path.isfile(artifact):
        print("artifact:", artifact, f"({os.path.getsize(artifact) / 1e6:.1f} MB)")
print()
print("Download these from the Kaggle 'Output' tab, then put the .npz in")
print("your local  models/graph/artifacts/  and point settings.lightgcn_artifact_path at it.")

## Done

- Read off **Recall@20 / NDCG@20 / MRR@20** from Step 4 for LightGCN vs Step 3 for popularity. LightGCN should win on every metric.
- Grab the `.npz` + `.json` from the **Output** tab. On a committed run they are under that version's Output tab.
- For the other domains: set `CATEGORY` in the config cell, then Save & Run All again. `PREPARED` is keyed by
  category + max_users, so switching categories will not reuse the wrong cache.

**If a run dies partway again**, check in this order:
1. Was it an interactive session? Use **Save Version -> Save & Run All (Commit)** instead.
2. GPU quota - Kaggle gives ~30 GPU-hours/week; the meter is on the Settings pane.
3. Out of memory - the log's last line will say so. Halve `MAX_USERS`, or drop `EVAL_BATCH` to 512.
4. Genuinely stuck - the elapsed/ETA lines print every 5 epochs; note the last one you saw and which step it was in.